In [2]:
import pandas as pd

In [6]:
df = pd.read_excel(r'C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2022_Dwarka-Sector_8_Delhi_DPCC__2022.xlsx')

In [7]:
df.head()

,Date,January,February,March,April,May,June,July,August,September,October,November,December
0,1,377.0,NaN,190.0,NaN,324.0,403.0,65.0,104.0,124.0,177,433.0,376.0
1,2,428.0,346.0,257.0,NaN,230.0,242.0,68.0,84.0,108.0,180,364.0,375.0
2,3,398.0,342.0,235.0,284.0,247.0,327.0,90.0,128.0,112.0,135,446.0,384.0
3,4,412.0,157.0,231.0,334.0,324.0,313.0,119.0,113.0,NaN,144,478.0,424.0
4,5,416.0,246.0,129.0,304.0,150.0,264.0,143.0,68.0,94.0,192,389.0,376.0


In [8]:
df.isnull().sum()

Date         0
January      1
February     5
March        3
April        3
May          2
June         3
July         1
August       5
September    3
October      0
November     1
December     1
dtype: int64

In [9]:
df

,Date,January,February,March,April,May,June,July,August,September,October,November,December
0,1,377.0,NaN,190.0,NaN,324.0,403.0,65.0,104.0,124.0,177,433.0,376.0
1,2,428.0,346.0,257.0,NaN,230.0,242.0,68.0,84.0,108.0,180,364.0,375.0
2,3,398.0,342.0,235.0,284.0,247.0,327.0,90.0,128.0,112.0,135,446.0,384.0
3,4,412.0,157.0,231.0,334.0,324.0,313.0,119.0,113.0,NaN,144,478.0,424.0
4,5,416.0,246.0,129.0,304.0,150.0,264.0,143.0,68.0,94.0,192,389.0,376.0
5,6,288.0,310.0,163.0,322.0,322.0,244.0,108.0,86.0,115.0,98,339.0,378.0
6,7,219.0,274.0,267.0,325.0,237.0,338.0,NaN,NaN,126.0,52,347.0,325.0
7,8,78.0,276.0,NaN,317.0,220.0,369.0,100.0,63.0,NaN,35,372.0,297.0
8,9,65.0,250.0,198.0,346.0,177.0,267.0,120.0,NaN,132.0,28,223.0,337.0
9,10,183.0,171.0,213.0,330.0,164.0,366.0,79.0,NaN,111.0,29,261.0,366.0


In [11]:
numeric_cols = df.select_dtypes(include=['number']).columns

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
print(df)

    Date     January    February   March       April         May        June  \
0      1  377.000000  246.153846  190.00  301.785714  324.000000  403.000000   
1      2  428.000000  346.000000  257.00  301.785714  230.000000  242.000000   
2      3  398.000000  342.000000  235.00  284.000000  247.000000  327.000000   
3      4  412.000000  157.000000  231.00  334.000000  324.000000  313.000000   
4      5  416.000000  246.000000  129.00  304.000000  150.000000  264.000000   
5      6  288.000000  310.000000  163.00  322.000000  322.000000  244.000000   
6      7  219.000000  274.000000  267.00  325.000000  237.000000  338.000000   
7      8   78.000000  276.000000  245.25  317.000000  220.000000  369.000000   
8      9   65.000000  250.000000  198.00  346.000000  177.000000  267.000000   
9     10  183.000000  171.000000  213.00  330.000000  164.000000  366.000000   
10    11  254.000000  244.000000  200.00  315.000000  151.000000  284.000000   
11    12  202.000000  200.000000  192.00

In [27]:
import calendar

# assume YEAR is defined (e.g., 2022)
YEAR = 2022

# Ensure Date is numeric (coerce strings like "01" -> 1)
df_melted["Date"] = pd.to_numeric(df_melted["Date"], errors="coerce").astype('Int64')

# Convert month names to month numbers (robust)
# If 'Month' is already numeric it will stay
if df_melted["Month"].dtype == object:
    # try parsing month names safely
    try:
        df_melted["Month_num"] = pd.to_datetime(df_melted["Month"], format="%B", errors="coerce").dt.month
    except Exception:
        df_melted["Month_num"] = pd.to_datetime(df_melted["Month"], errors="coerce").dt.month
else:
    df_melted["Month_num"] = df_melted["Month"].astype(int)

# If some Month_num are NaN (parsing failed), try matching first 3 letters (e.g., 'Jan')
mask_missing_month = df_melted["Month_num"].isna()
if mask_missing_month.any():
    df_melted.loc[mask_missing_month, "Month_num"] = df_melted.loc[mask_missing_month, "Month"].str[:3].map({
        'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12
    })

# Now compute safe day by capping to last day of month
def capped_day(row):
    day = int(row["Date"]) if pd.notna(row["Date"]) else 1
    month = int(row["Month_num"]) if pd.notna(row["Month_num"]) else 1
    last = calendar.monthrange(YEAR, month)[1]
    return min(day, last)

df_melted["CappedDay"] = df_melted.apply(capped_day, axis=1)

# Build Datetime (safe)
df_melted["Datetime"] = pd.to_datetime(
    df_melted.apply(lambda r: f"{YEAR:04d}-{int(r['Month_num']):02d}-{int(r['CappedDay']):02d}", axis=1),
    format="%Y-%m-%d",
    errors="coerce"
)

# Optionally report rows that were coerced to NaT (if any)
if df_melted["Datetime"].isna().any():
    print("Warning: some Datetime values could not be parsed and are NaT. Inspect these rows:")
    print(df_melted[df_melted["Datetime"].isna()][["Date","Month","Month_num","CappedDay"]])



In [28]:
df

,Date,January,February,March,April,May,June,July,August,September,October,November,December
0,1,377.000000,246.153846,190.00,301.785714,324.000000,403.000000,65.000000,104.000000,124.000000,177,433.000000,376.000000
1,2,428.000000,346.000000,257.00,301.785714,230.000000,242.000000,68.000000,84.000000,108.000000,180,364.000000,375.000000
2,3,398.000000,342.000000,235.00,284.000000,247.000000,327.000000,90.000000,128.000000,112.000000,135,446.000000,384.000000
3,4,412.000000,157.000000,231.00,334.000000,324.000000,313.000000,119.000000,113.000000,102.178571,144,478.000000,424.000000
4,5,416.000000,246.000000,129.00,304.000000,150.000000,264.000000,143.000000,68.000000,94.000000,192,389.000000,376.000000
5,6,288.000000,310.000000,163.00,322.000000,322.000000,244.000000,108.000000,86.000000,115.000000,98,339.000000,378.000000
6,7,219.000000,274.000000,267.00,325.000000,237.000000,338.000000,84.866667,85.692308,126.000000,52,347.000000,325.000000
7,8,78.000000,276.000000,245.25,317.000000,220.000000,369.000000,100.000000,63.000000,102.178571,35,372.000000,297.000000
8,9,65.000000,250.000000,198.00,346.000000,177.000000,267.000000,120.000000,85.692308,132.000000,28,223.000000,337.000000
9,10,183.000000,171.000000,213.00,330.000000,164.000000,366.000000,79.000000,85.692308,111.000000,29,261.000000,366.000000


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     int64  
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(11), int64(2)
memory usage: 3.3 KB


In [30]:
df.describe()

,Date,January,February,March,April,May,June,July,August,September,October,November,December
count,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000,31.000000
mean,16.000000,299.733333,246.153846,245.250000,301.785714,225.896552,228.178571,84.866667,85.692308,102.178571,214.870968,328.466667,341.733333
std,9.092121,90.490122,66.057900,49.953729,49.634905,67.427168,85.771811,26.927970,25.670696,39.291266,110.465000,65.740263,62.529957
min,1.000000,65.000000,90.000000,129.000000,188.000000,100.000000,92.000000,41.000000,37.000000,40.000000,28.000000,206.000000,155.000000
25%,8.500000,255.500000,214.000000,212.500000,287.000000,171.500000,153.000000,64.500000,67.000000,68.500000,138.000000,282.500000,327.000000
50%,16.000000,303.000000,246.153846,245.250000,314.000000,225.896552,237.000000,84.866667,85.692308,105.000000,249.000000,328.466667,351.000000
75%,23.500000,361.500000,293.500000,276.000000,335.500000,269.500000,277.000000,106.000000,103.500000,124.000000,281.000000,368.500000,377.000000
max,31.000000,428.000000,346.000000,343.000000,386.000000,364.000000,403.000000,143.000000,149.000000,215.000000,401.000000,478.000000,424.000000


In [31]:
print("\nMissing values per column:")
print(df.isnull().sum())



Missing values per column:
Date         0
January      0
February     0
March        0
April        0
May          0
June         0
July         0
August       0
September    0
October      0
November     0
December     0
dtype: int64


In [32]:
print("\nNumber of duplicate rows:")
print(df.duplicated().sum())



Number of duplicate rows:
0


In [33]:
import numpy as np

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] < lower) | (data[column] > upper)]

# Example: check outliers in January column
outliers_jan = detect_outliers_iqr(df, "January")
print(outliers_jan)


   Date  January  February   March  April    May   June   July     August  \
7     8     78.0     276.0  245.25  317.0  220.0  369.0  100.0  63.000000   
8     9     65.0     250.0  198.00  346.0  177.0  267.0  120.0  85.692308   

    September  October  November  December  
7  102.178571       35     372.0     297.0  
8  132.000000       28     223.0     337.0  


In [34]:
df

,Date,January,February,March,April,May,June,July,August,September,October,November,December
0,1,377.000000,246.153846,190.00,301.785714,324.000000,403.000000,65.000000,104.000000,124.000000,177,433.000000,376.000000
1,2,428.000000,346.000000,257.00,301.785714,230.000000,242.000000,68.000000,84.000000,108.000000,180,364.000000,375.000000
2,3,398.000000,342.000000,235.00,284.000000,247.000000,327.000000,90.000000,128.000000,112.000000,135,446.000000,384.000000
3,4,412.000000,157.000000,231.00,334.000000,324.000000,313.000000,119.000000,113.000000,102.178571,144,478.000000,424.000000
4,5,416.000000,246.000000,129.00,304.000000,150.000000,264.000000,143.000000,68.000000,94.000000,192,389.000000,376.000000
5,6,288.000000,310.000000,163.00,322.000000,322.000000,244.000000,108.000000,86.000000,115.000000,98,339.000000,378.000000
6,7,219.000000,274.000000,267.00,325.000000,237.000000,338.000000,84.866667,85.692308,126.000000,52,347.000000,325.000000
7,8,78.000000,276.000000,245.25,317.000000,220.000000,369.000000,100.000000,63.000000,102.178571,35,372.000000,297.000000
8,9,65.000000,250.000000,198.00,346.000000,177.000000,267.000000,120.000000,85.692308,132.000000,28,223.000000,337.000000
9,10,183.000000,171.000000,213.00,330.000000,164.000000,366.000000,79.000000,85.692308,111.000000,29,261.000000,366.000000
